In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('../../..')

In [ ]:
from PitchModel.Stuff.DataPrep.DataPrep import DataPrep
from PitchModel.Constants import DATA_PREP_BINARY_ALL_FILE

data_prep = DataPrep.Load_From_File("../../" + DATA_PREP_BINARY_ALL_FILE)

In [ ]:
pitch_list = data_prep.GenerateIOPitches(start_year=2022, end_year=2022)

In [ ]:
from PitchModel.Stuff.Tuning.TuneFunction import RunEvaluation
from PitchModel.Stuff.Model.ModelOutputType import ModelVariantType, ModelOutputType
from functools import partial

variant_type = ModelVariantType.Combined
output_type = ModelOutputType.InPlay
num_repeats = 3

objective_function = partial(
    RunEvaluation,
    pitch_list=pitch_list,
    data_prep=data_prep,
    model_variant_type=variant_type,
    model_output_type=output_type,
    max_repeats=num_repeats
)

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
study_name = f"{variant_type.name}_{output_type.name}"
#optuna.delete_study(study_name=study_name, storage="sqlite:///Tune.db")
study = optuna.create_study(
    direction="minimize",
    load_if_exists=True,
    study_name=study_name,
    storage="sqlite:///Tune.db"
)

In [ ]:
study.optimize(objective_function, n_trials=40, show_progress_bar=True)

In [ ]:
print(f"Final best value after refinement: {study.best_value:.4f}")
print(f"Best hyperparameters: {study.best_params}")

Create a study with reasonable values for viewing

In [ ]:
reasonable_value_dict = {
    (ModelVariantType.Stuff, ModelOutputType.Result) : 1,
    (ModelVariantType.Stuff, ModelOutputType.SwingResults) : 1.25,
    (ModelVariantType.Stuff, ModelOutputType.InPlay) : 1.9,
    (ModelVariantType.Combined, ModelOutputType.Result) : 0.6,
    (ModelVariantType.Combined, ModelOutputType.SwingResults) : 1,
    (ModelVariantType.Combined, ModelOutputType.InPlay) : 1.9,
}

In [ ]:
from optuna.trial import FrozenTrial, TrialState

def IsReasonable(trial : FrozenTrial):
    if trial.state != TrialState.COMPLETE or trial.value is None:
        return False
    return trial.value < num_repeats * reasonable_value_dict[(variant_type, output_type)]

In [ ]:
filtered_trials = [t for t in study.trials if IsReasonable(t)]
filtered_study = optuna.create_study(direction=study.direction)
for t in filtered_trials:
    filtered_study.add_trial(t)

View Results

In [ ]:
import optuna.visualization as vis
vis.plot_param_importances(filtered_study).show()

In [ ]:
vis.plot_optimization_history(filtered_study).show()

In [ ]:
vis.plot_slice(filtered_study).show()